# O-RAN Non-RT RIC: Energy Optimization LLM rApp
Run these cells sequentially to extract network metrics from Prometheus and pass them to the Cloud LLM for scaling decisions.

In [6]:
%pip install requests google-genai

Note: you may need to restart the kernel to use updated packages.


In [7]:
import requests
import json

PROMETHEUS_URL = "http://10.100.125.233:9090"

def get_prom_metric(query):
    """Helper function to fetch a single metric from Prometheus safely."""
    try:
        res = requests.get(f"{PROMETHEUS_URL}/api/v1/query", params={'query': query}, timeout=5).json()
        if res.get('status') == 'success' and res['data']['result']:
            return float(res['data']['result'][0]['value'][1])
    except Exception as e:
        print(f"Error fetching metric: {e}")
    return 0.0

def get_full_network_state():
    """Extracts CPU, UE counts, and Throughput to build the observation payload."""
    # 1. CPU Usage
    gnb_cpu = get_prom_metric('sum(rate(container_cpu_usage_seconds_total{namespace="open5gs", pod=~"srsran-gnb.*"}[1m]))')
    amf_cpu = get_prom_metric('sum(rate(container_cpu_usage_seconds_total{namespace="open5gs", pod=~"open5gs-amf.*"}[1m]))')
    
    # 2. UE Count (Direct from Open5GS AMF metrics!)
    active_ues = get_prom_metric('fivegs_amffunction_rm_registeredsubnbr')
    
    # 3. Throughput (Directly matching your Grafana UPF queries!)
    dl_mbps = get_prom_metric('sum(rate(container_network_receive_bytes_total{namespace="open5gs", pod=~"open5gs-upf.*"}[30s])) * 8 / 1e6')
    ul_mbps = get_prom_metric('sum(rate(container_network_transmit_bytes_total{namespace="open5gs", pod=~"open5gs-upf.*"}[30s])) * 8 / 1e6')
    
    payload = {
        "network_state": {
            "gNB_status": {
                "cpu_load_mcores": round(gnb_cpu * 1000, 2)
            },
            "Core_status": {
                "amf_cpu_load_mcores": round(amf_cpu * 1000, 2),
                "active_registered_ues": int(active_ues)
            },
            "UE_status": {
                "total_dl_throughput_mbps": round(dl_mbps, 3),
                "total_ul_throughput_mbps": round(ul_mbps, 3)
            }
        }
    }
    return payload

current_state = get_full_network_state()
print(json.dumps(current_state, indent=2))

{
  "network_state": {
    "gNB_status": {
      "cpu_load_mcores": 8.95
    },
    "Core_status": {
      "amf_cpu_load_mcores": 0.17,
      "active_registered_ues": 0
    },
    "UE_status": {
      "total_dl_throughput_mbps": 0.001,
      "total_ul_throughput_mbps": 0.003
    }
  }
}


In [ ]:
from google import genai
from google.genai import types
import time

# TODO: Replace with your actual API key
import os
try:
    with open('.env', 'r') as f:
        for line in f:
            if '=' in line and not line.startswith('#'):
                k, v = line.strip().split('=', 1)
                os.environ[k] = v
except FileNotFoundError:
    pass
API_KEY = os.environ.get('GEMINI_API_KEY')
if not API_KEY:
    raise ValueError('GEMINI_API_KEY is not set. Please set it in a .env file.')
client = genai.Client(api_key=API_KEY)

system_prompt = """
You are an O-RAN Non-RT RIC Energy Optimization AI. 
Your job is to read the JSON network_state provided to you and make a scaling decision for the gNB.
You must also provide a brief summary of the network state.

Rules for scaling:
1. If active_registered_ues is 0, or throughput is near 0, recommend scaling down to save energy.
2. If active_registered_ues > 0 and throughput is high, recommend scaling up or keeping current CPU.
3. NEVER recommend scaling the gNB CPU below 500m.

Return your response in pure JSON format matching this schema:
{"action": "scale_down" | "scale_up" | "do_nothing", "target_mcores": int, "network_summary": "string", "reasoning": "string"}
"""

def get_llm_decision(network_state):
    print("Asking the LLM for a decision...")
    # We wrap this in a retry loop to catch 'High Demand' 503 errors and auto-retry
    for attempt in range(4):
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f"Here is the current network state:\n{json.dumps(network_state)}",
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    response_mime_type="application/json"
                ),
            )
            return json.loads(response.text)
        except Exception as e:
            print(f"API is experiencing high demand (503 Error). Retrying in 5 seconds... (Attempt {attempt+1}/3)")
            time.sleep(5)
    return {"action": "do_nothing", "target_mcores": 500, "network_summary": "Error", "reasoning": "Failed to reach API after multiple attempts due to high demand."}

decision = get_llm_decision(current_state)
print("\n LLM Decision Output:")
print(json.dumps(decision, indent=2))

ValueError: GEMINI_API_KEY is not set. Please set it in a .env file.

In [ ]:
def apply_scaling_decision(target_mcores):
    print(f"Sending webhook to Host to dynamically scale cgroups to {target_mcores}m...")
    try:
        # 10.200.0.1 is the Host's bridge interface (cni0) visible to the pods
        res = requests.post("http://10.200.0.1:5000/scale", json={"target_mcores": target_mcores}, timeout=5)
        if res.status_code == 200:
            print("Successfully updated Linux cgroups on the fly (Zero Downtime)!")
            print(res.json())
        else:
            print(f"Webhook failed: {res.text}")
    except Exception as e:
        print(f"Failed to reach Host actuator. Make sure you ran 'sudo pip install flask' and 'sudo python3 actuator.py' on the VM: {e}")

if decision.get("action") in ["scale_down", "scale_up"]:
    apply_scaling_decision(decision["target_mcores"])
else:
    print("\U0001F4A4 No scaling action required. Maintaining current state.")

⚙️ Sending webhook to Host to dynamically scale cgroups to 500m...
✅ Successfully updated Linux cgroups on the fly (Zero Downtime)!
{'containers_patched': ['2ba5cb397d2ce6d69ea92fe7e51ccd6b3a8dbd0e49af149959ad51d6a63ee3a5'], 'new_mcores': 500, 'status': 'success'}
